# Preparación del entorno

El volumen Staging debe existir antes de ejecutar este Job.
Configuro catálogo, schema y volumen en el bloque 1.
La creación de infraestructura queda fuera de cada carga.


El notebook conserva Raw y usa Staging como copia derivada verificable.


In [ ]:
# BLOQUE 1 — Parámetros y referencia al snapshot Raw

import re

# Localizo los módulos en el repositorio completo, igual que el notebook Silver.
from pathlib import Path
import sys

dbutils.widgets.text("repo_root", "", "Raíz del repositorio en Workspace")
root_parameter = dbutils.widgets.get("repo_root").strip()
if root_parameter and not Path(root_parameter).is_absolute():
    raise ValueError("repo_root debe ser una ruta absoluta de Workspace.")
current = Path.cwd().resolve()
roots = [Path(root_parameter)] if root_parameter else [current, *current.parents]
project_root = next((p for p in roots if (p / "databricks/gtfs_bronze/runtime.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Sincroniza el repositorio completo y configura repo_root; falta gtfs_bronze.")
module_path = str(project_root / "databricks")
if module_path not in sys.path:
    sys.path.insert(0, module_path)

# Estos defaults corresponden a mi entorno actual; el Job puede reemplazarlos.
for key, default in {"catalog": "workspace", "schema": "gtfs_bogota"}.items():
    dbutils.widgets.text(key, default)
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
from gtfs_bronze.contracts import quoted_table
quoted_table(f"{catalog}.{schema}.configuration_check")
spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("raw_volume", "raw_gtfs")
raw_volume = dbutils.widgets.get("raw_volume").strip()
quoted_table(f"{catalog}.{schema}.{raw_volume}")
raw_volume_path = f"/Volumes/{catalog}/{schema}/{raw_volume}"
dbutils.widgets.text("staging_volume", "staging_gtfs")
staging_volume = dbutils.widgets.get("staging_volume").strip()
quoted_table(f"{catalog}.{schema}.{staging_volume}")
staging_volume_path = f"/Volumes/{catalog}/{schema}/{staging_volume}"

# El Job recibe estos valores desde la tarea Raw. Los defaults permiten
# reproducir interactivamente el snapshot ya validado de octubre de 2025.
dbutils.widgets.text("feed_version", "2025-10-28", "Versión GTFS")
dbutils.widgets.text(
    "snapshot_sha256",
    "07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952",
    "SHA-256 del snapshot",
)
feed_version = dbutils.widgets.get("feed_version").strip()
snapshot_sha256 = dbutils.widgets.get("snapshot_sha256").strip().lower()

if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", feed_version):
    raise ValueError("feed_version debe tener el formato AAAA-MM-DD.")
if not re.fullmatch(r"[0-9a-f]{64}", snapshot_sha256):
    raise ValueError("snapshot_sha256 debe ser un hash SHA-256 de 64 caracteres.")

raw_zip_path = (
    f"{raw_volume_path}/gtfs_static/"
    f"feed_version={feed_version}/"
    f"sha256={snapshot_sha256}/gtfs.zip"
)

print(f"ZIP Raw: {raw_zip_path}")
print(f"Staging: {staging_volume_path}")

display(dbutils.fs.ls(raw_zip_path.rsplit('/', 1)[0]))
dbutils.widgets.text("show_diagnostics", "false")
show_diagnostics_parameter = dbutils.widgets.get("show_diagnostics").strip().lower()
if show_diagnostics_parameter not in {"true", "false"}:
    raise ValueError("show_diagnostics debe ser true o false.")
show_diagnostics = show_diagnostics_parameter == "true"


In [ ]:
# BLOQUE 2 — Extracción idempotente de archivos GTFS hacia Staging

from datetime import datetime, timezone
import json
from pathlib import PurePosixPath
import uuid
import zipfile

from pyspark.sql.types import (
    BooleanType,
    LongType,
    StringType,
    StructField,
    StructType,
)

# Staging conserva una copia de trabajo derivada del ZIP Raw.
# La versión y el hash mantienen el vínculo con la evidencia original.
staging_snapshot_path = (
    f"{staging_volume_path}/gtfs_static/"
    f"feed_version={feed_version}/"
    f"sha256={snapshot_sha256}"
)

# Cada extracción tiene su propio identificador para auditoría.
extraction_run_id = str(uuid.uuid4())
extracted_at_utc = datetime.now(timezone.utc)

# Verifico la evidencia Raw antes de crear o reutilizar copias derivadas.
from gtfs_bronze.runtime import file_sha256, extract_verified
if file_sha256(raw_zip_path) != snapshot_sha256:
    raise RuntimeError("El ZIP Raw no coincide con el SHA-256 solicitado. No se modifica Raw ni Staging.")
dbutils.fs.mkdirs(staging_snapshot_path)

# Evitamos reescribir archivos que ya fueron extraídos para este snapshot.
existing_file_names = {
    file.name
    for file in dbutils.fs.ls(staging_snapshot_path)
}

extraction_rows = []

# Leemos el ZIP desde Raw. Nunca modificamos su contenido.
with open(raw_zip_path, "rb") as raw_zip_file:
    with zipfile.ZipFile(raw_zip_file) as gtfs_zip:

        # GTFS usa archivos de texto. Ignoramos directorios u otros archivos.
        members = [
            member
            for member in gtfs_zip.infolist()
            if not member.is_dir()
            and member.filename.lower().endswith(".txt")
        ]

        # Convertimos rutas internas del ZIP a nombres de archivo seguros.
        target_file_names = [
            PurePosixPath(member.filename).name
            for member in members
        ]

        # Evitamos sobrescribir datos si un ZIP tuviera dos archivos con el mismo nombre.
        if len(target_file_names) != len(set(target_file_names)):
            raise RuntimeError(
                "El ZIP contiene nombres de archivos GTFS duplicados."
            )

        for member, target_file_name in zip(members, target_file_names):
            staging_file_path = (
                f"{staging_snapshot_path}/{target_file_name}"
            )

            file_already_exists = (
                target_file_name in existing_file_names
            )

            # Un nombre existente no demuestra integridad del TXT.
            file_already_exists = extract_verified(gtfs_zip, member, staging_file_path)

            extraction_rows.append((
                member.filename,
                target_file_name,
                member.file_size,
                file_already_exists,
                staging_file_path,
            ))

# Usamos un esquema explícito para evitar que Spark infiera tipos ambiguos.
extraction_schema = StructType([
    StructField("source_member_name", StringType(), False),
    StructField("target_file_name", StringType(), False),
    StructField("uncompressed_bytes", LongType(), False),
    StructField("file_already_exists", BooleanType(), False),
    StructField("staging_file_path", StringType(), False),
])

extraction_df = spark.createDataFrame(
    extraction_rows,
    schema=extraction_schema,
)

# Guardamos un manifiesto de la extracción derivada.
extraction_manifest = {
    "extraction_run_id": extraction_run_id,
    "extracted_at_utc": extracted_at_utc.isoformat(),
    "source_raw_zip_path": raw_zip_path,
    "feed_version": feed_version,
    "source_sha256": snapshot_sha256,
    "staging_snapshot_path": staging_snapshot_path,
    "files_extracted": [
        {
            "source_member_name": row[0],
            "target_file_name": row[1],
            "uncompressed_bytes": row[2],
            "file_already_exists": row[3],
        }
        for row in extraction_rows
    ],
}

extraction_manifest_path = (
    f"{staging_snapshot_path}/"
    f"_extraction_run_id={extraction_run_id}.json"
)

dbutils.fs.put(
    extraction_manifest_path,
    json.dumps(extraction_manifest, indent=2),
    overwrite=False,
)

display(extraction_df)

In [ ]:
# BLOQUE 3 — Lectura y prevalidación de routes.txt desde Staging

from pyspark.sql import functions as F
from gtfs_bronze.runtime import read_gtfs_header, source_frame, load_frame
routes_staging_path = f"{staging_snapshot_path}/routes.txt"
routes_columns = read_gtfs_header(routes_staging_path)
required_routes_columns = {"route_id", "route_short_name", "route_long_name", "route_type"}
missing_routes_columns = sorted(required_routes_columns - set(routes_columns))
if missing_routes_columns:
    raise RuntimeError(f"routes.txt incumple su contrato mínimo: {missing_routes_columns}")
routes_source_df = source_frame(spark, routes_staging_path, routes_columns)
# La procedencia se añade en el cargador; no repito aquí conteos de la misma fuente.


In [ ]:
# BLOQUE 4 — Escritura idempotente de routes.txt en Delta Bronze

bronze_routes_table = f"{catalog}.{schema}.bronze_gtfs_routes"
routes_source_count, snapshot_already_loaded = load_frame(
    spark, routes_source_df, table=bronze_routes_table, source_path=routes_staging_path,
    feed_version=feed_version, sha256=snapshot_sha256,
)
bronze_snapshot_df = spark.table(bronze_routes_table).filter(F.col("source_sha256") == snapshot_sha256)
print(f"Tabla Bronze: {bronze_routes_table}; filas verificadas: {routes_source_count}; reutilizado: {snapshot_already_loaded}")


In [ ]:
# Diagnóstico opcional; notebook 07 mantiene la compuerta obligatoria.
if show_diagnostics:
    # BLOQUE 4A — Auditoría de deriva de esquema entre snapshots de rutas

    from pyspark.sql.types import StringType, StructField, StructType

    # Las columnas técnicas las agrega el pipeline; la comparación se limita
    # a los campos publicados por TransMilenio en routes.txt.
    technical_columns = {
        "source_feed_version",
        "source_sha256",
        "source_file_path",
        "bronze_loaded_at_utc",
    }
    bronze_data_columns = (
        set(spark.table(bronze_routes_table).columns) - technical_columns
    )
    source_columns = set(routes_source_df.columns)

    # Identificamos cambios del proveedor sin añadir valores ni eliminar historia.
    missing_in_current_source = sorted(bronze_data_columns - source_columns)
    new_in_current_source = sorted(source_columns - bronze_data_columns)

    schema_drift_schema = StructType([
        StructField("drift_type", StringType(), False),
        StructField("columns", StringType(), False),
    ])
    schema_drift_rows = [
        (
            "missing_in_current_source",
            ", ".join(missing_in_current_source) or "ninguna",
        ),
        (
            "new_in_current_source",
            ", ".join(new_in_current_source) or "ninguna",
        ),
    ]

    display(
        spark.createDataFrame(
            schema_drift_rows,
            schema=schema_drift_schema,
        )
    )

    # Esta métrica comprueba cómo la columna histórica se comporta por versión.
    # Un valor nulo en 2026 es evidencia de ausencia en el proveedor, no un valor inventado.
    if "route_desc" in spark.table(bronze_routes_table).columns:
        display(
            spark.table(bronze_routes_table)
            .groupBy("source_feed_version")
            .agg(
                F.count("*").alias("route_records"),
                F.sum(
                    F.when(F.col("route_desc").isNotNull(), 1).otherwise(0)
                ).alias("route_desc_non_null_records"),
            )
            .orderBy("source_feed_version")
        )
    else:
        print("La tabla histórica no contiene route_desc; este diagnóstico no aplica.")


In [ ]:
# Diagnóstico opcional; notebook 07 mantiene la compuerta obligatoria.
if show_diagnostics:
    # Consulta las primeras rutas persistidas en Bronze.
    display(
        spark.table(bronze_routes_table)
        .orderBy("route_id")
        .limit(20)
    )


In [ ]:
# Diagnóstico opcional; notebook 07 mantiene la compuerta obligatoria.
if show_diagnostics:
    # Muestra formato, ubicación, número de archivos y tamaño de la tabla.
    display(
        spark.sql(f"""
            describe detail {catalog}.{schema}.bronze_gtfs_routes
        """)
    )


In [ ]:
# Diagnóstico opcional; notebook 07 mantiene la compuerta obligatoria.
if show_diagnostics:
    # Muestra las versiones y operaciones realizadas sobre la tabla.
    display(
        spark.sql(f"""
            describe history {catalog}.{schema}.bronze_gtfs_routes
        """)
    )
